# 04-1. Location Ablation Experiment

Evaluate selected location feature candidates across Linear Regression, Decision Tree, and Random Forest using the same 18-month rolling validation splits as the baseline notebooks.

The current default run is Stage 3 candidate validation:
- Linear Regression candidates focus on `PostalCode`, `MLSAreaMajor`, and `City`.
- Decision Tree and Random Forest candidates focus on `Latitude`/`Longitude` plus compact categorical add-ons.
- `C0_BASE` and `C7_FULL_NO_FLAGS` are run for every model as common reference points.

Outputs are saved under `outputs/location_ablation/`.

## 1. Setup

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 160)
sns.set_theme(style="whitegrid")

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "idx-california-price-prediction",
]
PROJECT_ROOT = next(
    root for root in candidate_roots
    if (root / "data" / "processed").exists() and (root / "notebooks").exists()
)
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = PROCESSED_DATA_DIR / "splits"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "location_ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_PLAN_CSV = PROCESSED_DATA_DIR / "crmls_week3_split_plan.csv"
TARGET = "ClosePrice"
RANDOM_STATE = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Split directory: {SPLIT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Experiment Configuration

`HighSchoolDistrict` is kept in the non-location base by default because it is part of the current baseline categorical feature set but is not one of the location bundles tested here. Set `INCLUDE_HIGH_SCHOOL_DISTRICT_IN_BASE = False` to remove it from the base model.

In [ ]:
RUN_EXPERIMENT = True
INCLUDE_HIGH_SCHOOL_DISTRICT_IN_BASE = True

MODELS_TO_RUN = [
    "LinearRegression",
    "DecisionTreeRegressor",
    "RandomForestRegressor",
]
EVAL_MONTHS_TO_RUN = None

EXPERIMENTS_TO_RUN = None
# Example for a quick smoke test:
# MODELS_TO_RUN = ["LinearRegression"]
# EVAL_MONTHS_TO_RUN = ["2026-05"]
# EXPERIMENTS_TO_RUN = ["L0_BASE", "L6_FULL_NO_FLAGS", "L12_FULL_WITH_FLAGS"]

HIGH_CARDINALITY_CATEGORICAL_COLS = ["City", "PostalCode", "MLSAreaMajor"]
HIGH_CARDINALITY_MIN_FREQUENCY = 20
CATEGORICAL_DTYPE_COLS = ["City", "CountyOrParish", "PostalCode", "MLSAreaMajor", "Levels", "HighSchoolDistrict"]

LOCATION_BUNDLES = {
    "COORD": {"numeric": ["Latitude", "Longitude"], "categorical": [], "flag": []},
    "COUNTY": {"numeric": [], "categorical": ["CountyOrParish"], "flag": []},
    "CITY": {"numeric": [], "categorical": ["City"], "flag": []},
    "POSTAL": {"numeric": [], "categorical": ["PostalCode"], "flag": []},
    "MLS_AREA": {"numeric": [], "categorical": ["MLSAreaMajor"], "flag": []},
}

LOCATION_FLAGS = [
    "Latitude_missing",
    "Longitude_missing",
    "invalid_coordinates_flag",
    "PostalCode_format_issue_flag",
]

BASE_CONTINUOUS_NUMERIC_COLS = [
    "LivingArea",
    "LotSizeSquareFeet",
    "YearBuilt",
    "AssociationFee",
]
BASE_COUNT_NUMERIC_COLS = [
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "GarageSpaces",
    "ParkingTotal",
    "Stories",
    "Flooring_material_count",
]
BASE_NUMERIC_COLS = BASE_CONTINUOUS_NUMERIC_COLS + BASE_COUNT_NUMERIC_COLS
BASE_BOOLEAN_COLS = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN",
    "Flooring_Carpet",
    "Flooring_Tile",
    "Flooring_Wood",
    "Flooring_Laminate",
    "Flooring_Vinyl",
    "Flooring_Stone",
    "Flooring_Concrete",
    "Flooring_Bamboo",
    "Flooring_Brick",
    "Flooring_SeeRemarks",
    "Flooring_multiple_materials_flag",
]
BASE_CATEGORICAL_COLS = ["Levels"]
if INCLUDE_HIGH_SCHOOL_DISTRICT_IN_BASE:
    BASE_CATEGORICAL_COLS.append("HighSchoolDistrict")

## 3. Load Split Plan

In [ ]:
split_plan = pd.read_csv(SPLIT_PLAN_CSV, dtype={"split_type": "string", "eval_month": "string"})
historical_plan = (
    split_plan[split_plan["split_type"] == "historical_eval"]
    .copy()
    .sort_values("eval_month")
)
if EVAL_MONTHS_TO_RUN is not None:
    historical_plan = historical_plan[
        historical_plan["eval_month"].astype(str).isin(set(EVAL_MONTHS_TO_RUN))
    ].copy()
    if historical_plan.empty:
        raise ValueError(f"No historical folds match EVAL_MONTHS_TO_RUN={EVAL_MONTHS_TO_RUN}")
final_plan = split_plan[split_plan["split_type"] == "final_test"].copy().iloc[0]

print(f"Historical cutoffs: {historical_plan['eval_month'].astype(str).tolist()}")
print(f"Final train window months: {int(final_plan['train_window_months'])}")
display(historical_plan[["eval_month", "train_window_months", "train_month_start", "train_month_end", "train_rows", "eval_rows"]])

## 4. Location Feature Sets

In [ ]:
def merge_bundle_names(bundle_names, include_location_flags=False):
    config = {"numeric": [], "categorical": [], "boolean": [], "flag": []}
    for bundle_name in bundle_names:
        bundle = LOCATION_BUNDLES[bundle_name]
        for group in ["numeric", "categorical", "flag"]:
            config[group].extend(bundle[group])
    if include_location_flags:
        config["flag"].extend(LOCATION_FLAGS)
    return config


def unique_preserve_order(values):
    return list(dict.fromkeys(values))


EXPERIMENT_DEFINITIONS = [
    {
        "experiment_id": "C0_BASE",
        "stage": "stage_3_candidate",
        "bundles": [],
        "include_location_flags": False,
        "models": ["LinearRegression", "DecisionTreeRegressor", "RandomForestRegressor"],
    },
    {
        "experiment_id": "C1_LR_POSTAL",
        "stage": "stage_3_candidate",
        "bundles": ["POSTAL"],
        "include_location_flags": False,
        "models": ["LinearRegression"],
    },
    {
        "experiment_id": "C2_LR_POSTAL_MLS_AREA",
        "stage": "stage_3_candidate",
        "bundles": ["POSTAL", "MLS_AREA"],
        "include_location_flags": False,
        "models": ["LinearRegression"],
    },
    {
        "experiment_id": "C3_LR_POSTAL_CITY_MLS_AREA",
        "stage": "stage_3_candidate",
        "bundles": ["POSTAL", "CITY", "MLS_AREA"],
        "include_location_flags": False,
        "models": ["LinearRegression"],
    },
    {
        "experiment_id": "C4_TREE_COORD",
        "stage": "stage_3_candidate",
        "bundles": ["COORD"],
        "include_location_flags": False,
        "models": ["DecisionTreeRegressor", "RandomForestRegressor"],
    },
    {
        "experiment_id": "C5_TREE_COORD_MLS_AREA",
        "stage": "stage_3_candidate",
        "bundles": ["COORD", "MLS_AREA"],
        "include_location_flags": False,
        "models": ["DecisionTreeRegressor", "RandomForestRegressor"],
    },
    {
        "experiment_id": "C6_TREE_COORD_POSTAL",
        "stage": "stage_3_candidate",
        "bundles": ["COORD", "POSTAL"],
        "include_location_flags": False,
        "models": ["DecisionTreeRegressor", "RandomForestRegressor"],
    },
    {
        "experiment_id": "C7_FULL_NO_FLAGS",
        "stage": "stage_3_candidate",
        "bundles": ["COORD", "COUNTY", "CITY", "POSTAL", "MLS_AREA"],
        "include_location_flags": False,
        "models": ["LinearRegression", "DecisionTreeRegressor", "RandomForestRegressor"],
    },
]

if EXPERIMENTS_TO_RUN is not None:
    EXPERIMENT_DEFINITIONS = [
        experiment for experiment in EXPERIMENT_DEFINITIONS
        if experiment["experiment_id"] in set(EXPERIMENTS_TO_RUN)
    ]

experiment_overview = pd.DataFrame(EXPERIMENT_DEFINITIONS)
experiment_overview["location_features"] = experiment_overview["bundles"].apply(lambda bundles: ", ".join(bundles) if bundles else "none")
experiment_overview["models_to_run"] = experiment_overview["models"].apply(lambda models: ", ".join(models))
display(experiment_overview[["experiment_id", "stage", "location_features", "include_location_flags", "models_to_run"]])

## 5. Preprocessing and Evaluation Helpers

In [ ]:
def normalize_categorical_dtypes(df):
    for col in CATEGORICAL_DTYPE_COLS:
        if col in df.columns:
            df[col] = df[col].astype("string").fillna("__missing__")
    return df


def load_split_file(path):
    df = normalize_categorical_dtypes(pd.read_csv(path, low_memory=False))
    if "CloseDate" in df.columns:
        df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
    if "close_month" in df.columns:
        df["close_month"] = pd.PeriodIndex(df["close_month"], freq="M")
    return df


def rolling_train_path(eval_month):
    return SPLIT_DIR / f"rolling_train_eval_{eval_month}_cleaned.csv"


def rolling_eval_path(eval_month):
    return SPLIT_DIR / f"rolling_eval_{eval_month}_cleaned.csv"


def available(columns, df):
    return [col for col in unique_preserve_order(columns) if col in df.columns]


def build_feature_groups(train_df, experiment):
    location_config = merge_bundle_names(
        experiment["bundles"],
        include_location_flags=experiment["include_location_flags"],
    )
    base_flag_cols = [
        col for col in train_df.columns
        if (col.endswith("_missing") or col == "invalid_coordinates_flag")
        and col not in LOCATION_FLAGS
    ]
    groups = {
        "numeric": available(BASE_NUMERIC_COLS + location_config["numeric"], train_df),
        "categorical": available(BASE_CATEGORICAL_COLS + location_config["categorical"], train_df),
        "boolean": available(BASE_BOOLEAN_COLS, train_df),
        "flag": available(base_flag_cols + location_config["flag"], train_df),
    }
    groups["feature_cols"] = groups["numeric"] + groups["categorical"] + groups["boolean"] + groups["flag"]
    groups["location_feature_cols"] = available(
        location_config["numeric"] + location_config["categorical"] + location_config["flag"],
        train_df,
    )
    return groups


def make_regular_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def make_high_cardinality_encoder(min_frequency=HIGH_CARDINALITY_MIN_FREQUENCY):
    if min_frequency is None:
        return make_regular_onehot_encoder()
    try:
        return OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=min_frequency,
            sparse_output=True,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=min_frequency,
            sparse=True,
        )


def split_categorical_features(categorical_features):
    high_cardinality_features = [
        col for col in categorical_features
        if col in HIGH_CARDINALITY_CATEGORICAL_COLS
    ]
    regular_categorical_features = [
        col for col in categorical_features
        if col not in HIGH_CARDINALITY_CATEGORICAL_COLS
    ]
    return high_cardinality_features, regular_categorical_features


def build_preprocessing_pipeline(feature_groups, scale_numeric=True):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipeline = Pipeline(steps=numeric_steps)
    high_cardinality_features, regular_categorical_features = split_categorical_features(feature_groups["categorical"])
    high_cardinality_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("onehot", make_high_cardinality_encoder()),
    ])
    regular_categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("onehot", make_regular_onehot_encoder()),
    ])
    boolean_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent"))])
    flag_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="constant", fill_value=0))])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, feature_groups["numeric"]),
            ("high_cat", high_cardinality_pipeline, high_cardinality_features),
            ("regular_cat", regular_categorical_pipeline, regular_categorical_features),
            ("bool", boolean_pipeline, feature_groups["boolean"]),
            ("flag", flag_pipeline, feature_groups["flag"]),
        ],
        remainder="drop",
        sparse_threshold=0.3,
        verbose_feature_names_out=True,
    )


def build_model_pipeline(estimator, feature_groups, scale_numeric=True):
    return Pipeline(steps=[
        ("preprocess", build_preprocessing_pipeline(feature_groups, scale_numeric=scale_numeric)),
        ("model", estimator),
    ])

In [ ]:
def apply_train_price_bounds(train_df, eval_df, target_col=TARGET, lower_q=0.005, upper_q=0.995):
    lower_bound = train_df[target_col].quantile(lower_q)
    upper_bound = train_df[target_col].quantile(upper_q)
    train_mask = train_df[target_col].between(lower_bound, upper_bound)
    eval_mask = eval_df[target_col].between(lower_bound, upper_bound)
    return (
        train_df.loc[train_mask].copy(),
        eval_df.loc[eval_mask].copy(),
        {
            "price_lower_bound": lower_bound,
            "price_upper_bound": upper_bound,
            "train_rows_before_outlier_filter": len(train_df),
            "train_rows_after_outlier_filter": int(train_mask.sum()),
            "eval_rows_before_outlier_filter": len(eval_df),
            "eval_rows_after_outlier_filter": int(eval_mask.sum()),
        },
    )


def regression_metrics(y_true, y_pred):
    y_true = pd.Series(y_true).astype(float).reset_index(drop=True)
    y_pred = pd.Series(y_pred).astype(float).reset_index(drop=True)
    ape = ((y_true - y_pred).abs() / y_true).replace([np.inf, -np.inf], np.nan) * 100
    return {
        "r2": r2_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape": ape.mean(),
        "mdape": ape.median(),
    }


MODEL_SPECS = {
    "LinearRegression": {
        "estimator_factory": lambda: LinearRegression(),
        "scale_numeric": True,
    },
    "DecisionTreeRegressor": {
        "estimator_factory": lambda: DecisionTreeRegressor(random_state=RANDOM_STATE),
        "scale_numeric": False,
    },
    "RandomForestRegressor": {
        "estimator_factory": lambda: RandomForestRegressor(
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "scale_numeric": False,
    },
}

MODEL_SPECS = {name: spec for name, spec in MODEL_SPECS.items() if name in set(MODELS_TO_RUN)}
display(pd.DataFrame({"model": list(MODEL_SPECS)}))

## 6. Run Rolling Ablation

In [ ]:
results = []

if RUN_EXPERIMENT:
    total_fits = len(historical_plan) * sum(
        len(set(experiment["models"]) & set(MODEL_SPECS))
        for experiment in EXPERIMENT_DEFINITIONS
    )
    fit_number = 0
    started_at = time.time()

    for _, plan_row in historical_plan.iterrows():
        eval_month = str(plan_row["eval_month"])
        train_df_raw = load_split_file(rolling_train_path(eval_month))
        eval_df_raw = load_split_file(rolling_eval_path(eval_month))
        train_df, eval_df, bound_info = apply_train_price_bounds(train_df_raw, eval_df_raw)

        for experiment in EXPERIMENT_DEFINITIONS:
            feature_groups = build_feature_groups(train_df, experiment)
            for model_name, spec in MODEL_SPECS.items():
                if model_name not in set(experiment["models"]):
                    continue
                fit_number += 1
                print(
                    f"[{fit_number}/{total_fits}] {eval_month} | {experiment['experiment_id']} | {model_name}",
                    flush=True,
                )
                pipeline = build_model_pipeline(
                    spec["estimator_factory"](),
                    feature_groups,
                    scale_numeric=spec["scale_numeric"],
                )
                pipeline.fit(train_df[feature_groups["feature_cols"]], train_df[TARGET])
                train_predictions = pipeline.predict(train_df[feature_groups["feature_cols"]])
                eval_predictions = pipeline.predict(eval_df[feature_groups["feature_cols"]])
                train_metrics = regression_metrics(train_df[TARGET], train_predictions)
                eval_metrics = regression_metrics(eval_df[TARGET], eval_predictions)
                encoded_feature_count = len(pipeline.named_steps["preprocess"].get_feature_names_out())

                results.append({
                    "model": model_name,
                    "experiment_id": experiment["experiment_id"],
                    "stage": experiment["stage"],
                    "location_bundles": ",".join(experiment["bundles"]) if experiment["bundles"] else "none",
                    "location_features": ",".join(feature_groups["location_feature_cols"]) if feature_groups["location_feature_cols"] else "none",
                    "location_flags_used": experiment["include_location_flags"],
                    "candidate_models": ",".join(experiment["models"]),
                    "high_cardinality_min_frequency": HIGH_CARDINALITY_MIN_FREQUENCY,
                    "train_window_months": int(plan_row["train_window_months"]),
                    "eval_month": eval_month,
                    "train_month_start": plan_row["train_month_start"],
                    "train_month_end": plan_row["train_month_end"],
                    "train_months": plan_row["train_months"],
                    **bound_info,
                    "n_raw_features": len(feature_groups["feature_cols"]),
                    "encoded_feature_count": encoded_feature_count,
                    "train_r2": train_metrics["r2"],
                    "validation_r2": eval_metrics["r2"],
                    "train_validation_r2_gap": train_metrics["r2"] - eval_metrics["r2"],
                    "validation_mae": eval_metrics["mae"],
                    "validation_rmse": eval_metrics["rmse"],
                    "validation_mape": eval_metrics["mape"],
                    "validation_mdape": eval_metrics["mdape"],
                    "elapsed_minutes": (time.time() - started_at) / 60,
                })

    location_ablation_results = pd.DataFrame(results)
    location_ablation_results.to_csv(OUTPUT_DIR / "location_ablation_results.csv", index=False)
else:
    location_ablation_results = pd.read_csv(OUTPUT_DIR / "location_ablation_results.csv")

display(location_ablation_results.head())
print(f"Rows: {len(location_ablation_results):,}")

## 7. Summarize Results

In [ ]:
summary = (
    location_ablation_results
    .groupby(["model", "experiment_id", "stage", "location_bundles", "location_features", "location_flags_used"], sort=False)
    .agg(
        mean_validation_r2=("validation_r2", "mean"),
        std_validation_r2=("validation_r2", "std"),
        mean_validation_mae=("validation_mae", "mean"),
        mean_validation_rmse=("validation_rmse", "mean"),
        mean_validation_mdape=("validation_mdape", "mean"),
        mean_encoded_feature_count=("encoded_feature_count", "mean"),
        mean_train_validation_r2_gap=("train_validation_r2_gap", "mean"),
    )
    .reset_index()
)

base_metrics = summary[summary["experiment_id"] == "C0_BASE"][[
    "model",
    "mean_validation_r2",
    "mean_validation_mae",
    "mean_validation_mdape",
    "mean_encoded_feature_count",
]].rename(columns={
    "mean_validation_r2": "base_mean_validation_r2",
    "mean_validation_mae": "base_mean_validation_mae",
    "mean_validation_mdape": "base_mean_validation_mdape",
    "mean_encoded_feature_count": "base_mean_encoded_feature_count",
})

full_metrics = summary[summary["experiment_id"] == "C7_FULL_NO_FLAGS"][[
    "model",
    "mean_validation_r2",
    "mean_validation_mae",
    "mean_validation_mdape",
    "mean_encoded_feature_count",
]].rename(columns={
    "mean_validation_r2": "full_no_flags_mean_validation_r2",
    "mean_validation_mae": "full_no_flags_mean_validation_mae",
    "mean_validation_mdape": "full_no_flags_mean_validation_mdape",
    "mean_encoded_feature_count": "full_no_flags_mean_encoded_feature_count",
})

summary = summary.merge(base_metrics, on="model", how="left").merge(full_metrics, on="model", how="left")
summary["mean_validation_r2_delta_vs_base"] = summary["mean_validation_r2"] - summary["base_mean_validation_r2"]
summary["mean_validation_mae_delta_vs_base"] = summary["mean_validation_mae"] - summary["base_mean_validation_mae"]
summary["mean_validation_mdape_delta_vs_base"] = summary["mean_validation_mdape"] - summary["base_mean_validation_mdape"]
summary["encoded_feature_count_delta_vs_base"] = summary["mean_encoded_feature_count"] - summary["base_mean_encoded_feature_count"]
summary["mean_validation_r2_delta_vs_full_no_flags"] = summary["mean_validation_r2"] - summary["full_no_flags_mean_validation_r2"]
summary["mean_validation_mae_delta_vs_full_no_flags"] = summary["mean_validation_mae"] - summary["full_no_flags_mean_validation_mae"]
summary["mean_validation_mdape_delta_vs_full_no_flags"] = summary["mean_validation_mdape"] - summary["full_no_flags_mean_validation_mdape"]
summary["encoded_feature_count_delta_vs_full_no_flags"] = summary["mean_encoded_feature_count"] - summary["full_no_flags_mean_encoded_feature_count"]

summary = summary.sort_values(["model", "mean_validation_r2"], ascending=[True, False])
summary.to_csv(OUTPUT_DIR / "location_ablation_summary.csv", index=False)

display(summary[[
    "model",
    "experiment_id",
    "stage",
    "location_bundles",
    "location_flags_used",
    "mean_validation_r2",
    "std_validation_r2",
    "mean_validation_r2_delta_vs_base",
    "mean_validation_r2_delta_vs_full_no_flags",
    "mean_validation_mdape",
    "mean_encoded_feature_count",
    "mean_train_validation_r2_gap",
]].round(4))

## 8. Stage-Specific Tables

In [ ]:
candidate_summary = summary[summary["stage"] == "stage_3_candidate"].copy()

print("Stage 3: model-specific candidate ranking")
display(candidate_summary[[
    "model",
    "experiment_id",
    "location_bundles",
    "mean_validation_r2",
    "std_validation_r2",
    "mean_validation_r2_delta_vs_base",
    "mean_validation_r2_delta_vs_full_no_flags",
    "mean_validation_mdape",
    "mean_encoded_feature_count",
    "mean_train_validation_r2_gap",
]].sort_values(["model", "mean_validation_r2", "mean_validation_mdape"], ascending=[True, False, True]).round(4))

best_candidate_by_model = (
    candidate_summary
    .sort_values(["model", "mean_validation_r2", "mean_validation_mdape"], ascending=[True, False, True])
    .groupby("model")
    .head(1)
)

print("Best candidate by model")
display(best_candidate_by_model[[
    "model",
    "experiment_id",
    "location_bundles",
    "mean_validation_r2",
    "std_validation_r2",
    "mean_validation_mdape",
    "mean_encoded_feature_count",
]].round(4))

## 9. Visual Checks

In [ ]:
plot_df = summary.copy()
order = [experiment["experiment_id"] for experiment in EXPERIMENT_DEFINITIONS]

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

sns.barplot(
    data=plot_df,
    x="experiment_id",
    y="mean_validation_r2",
    hue="model",
    order=order,
    ax=axes[0],
)
axes[0].set_title("Mean Rolling Validation R2 by Location Ablation")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean validation R2")
axes[0].legend(loc="lower right")

sns.barplot(
    data=plot_df,
    x="experiment_id",
    y="mean_encoded_feature_count",
    hue="model",
    order=order,
    ax=axes[1],
)
axes[1].set_title("Encoded Feature Count by Location Ablation")
axes[1].set_xlabel("Experiment")
axes[1].set_ylabel("Mean encoded feature count")
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

## 10. Candidate Selection Notes

Use Stage 1 to identify standalone location value and Stage 2 to identify incremental value after other location signals are already present.

After reviewing the tables, define 2-4 compact candidate feature sets for a follow-up Stage 3 experiment. Keep model choices separate if Linear Regression, Decision Tree, and Random Forest prefer different location representations.

In [ ]:
best_by_model = (
    summary.sort_values(["model", "mean_validation_r2", "mean_validation_mdape"], ascending=[True, False, True])
    .groupby("model")
    .head(5)
)

display(best_by_model[[
    "model",
    "experiment_id",
    "stage",
    "location_bundles",
    "location_flags_used",
    "mean_validation_r2",
    "std_validation_r2",
    "mean_validation_r2_delta_vs_base",
    "mean_validation_mdape",
    "mean_encoded_feature_count",
]].round(4))

print(f"Saved detailed results: {OUTPUT_DIR / 'location_ablation_results.csv'}")
print(f"Saved summary: {OUTPUT_DIR / 'location_ablation_summary.csv'}")